### DECS<->VISA example

Here a demonstration of working with DECS<->VISA from a jupyter notebook is provided.

The general operation is very similar to the `example_client.py` code, but rather than require a terminal be opened to launch `decs_visa.py` we will do it directly from the notebook using a python subprocess.

*Note: When running on Windows, outputs from DECS<->VISA are stored in a `decs_visa.log` file, which will be created in your working directory. If struggling to establish a connection between DECS<->VISA and oi.DECS, check the information captured in the `decs_visa.log` file.*

In [1]:
import time
import subprocess
import platform
import Proteox_helpers as ph

decs_visa_path = './decs_visa.py'

# Start the decs_visa subprocess
running_on = platform.platform()
if running_on.startswith("Windows"):
    print(f"Running on {running_on} - start subprocess without PIPEd output")
    subprocess.Popen(["python", decs_visa_path])
else:
    print(f"Running on {running_on} - start subprocess with PIPEd output")
    subprocess.Popen(["python3", decs_visa_path], stdout=subprocess.PIPE)
    
# short delay to allow for the server to open
time.sleep(1)

Running on Windows-10-10.0.26200-SP0 - start subprocess without PIPEd output


Once the server is open we can connect to it in the same way as the `client_example.py` code - as that has already worked, we'll skip some of the error checking here for brevity.

In [2]:
import pyvisa as visa

rm = visa.ResourceManager('@py')
decs_visa_server_ip = "localhost"
decs_visa_server_port = "33576"
pyvisa_connection = f"TCPIP0::{decs_visa_server_ip}::{decs_visa_server_port}::SOCKET"
decs_visa = rm.open_resource(pyvisa_connection)
decs_visa.read_termination = "\n"
decs_visa.write_termination = "\n"
decs_visa.chunk_size = 204800
decs_visa.timeout = 10000


try:
    print(decs_visa.query("*IDN?"))
except Exception as e:
    print(e)

QD - Oxford, DECS, decs-55fff5, 1.7.1.8747


In [3]:
# get temperature
decs_visa.query("get_MC_T")

'0.0110408'

In [4]:
# change still heater setpoint
try:
    ph.set_still_heater_power(decs_visa, 10e-3)  # example: 100 uW

    for i in range(60):
        T_mK = float(decs_visa.query("get_MC_T")) * 1000
        T_still_mK = float(decs_visa.query("get_STILL_T")) * 1000
        # flow = float(decs_visa.query("get_3He_F"))
        print(f"MC T = {T_mK:.2f} mK")
        print(f"Still T = {T_still_mK:.2f} mK")
        # print(f"He3 Flow = {flow:.2f}")
        time.sleep(60)

finally:
    ph.all_heaters_off(decs_visa)

MC T = 49.91 mK
Still T = 624.10 mK
MC T = 47.97 mK
Still T = 779.80 mK
MC T = 46.67 mK
Still T = 789.80 mK
MC T = 46.87 mK
Still T = 799.00 mK
MC T = 47.94 mK
Still T = 806.30 mK
MC T = 48.97 mK
Still T = 814.00 mK
MC T = 49.78 mK
Still T = 821.90 mK
MC T = 50.21 mK
Still T = 831.20 mK
MC T = 50.30 mK
Still T = 841.50 mK
MC T = 50.30 mK
Still T = 856.60 mK
MC T = 50.20 mK
Still T = 870.80 mK
MC T = 50.26 mK
Still T = 884.80 mK
MC T = 50.26 mK
Still T = 899.20 mK
MC T = 50.24 mK
Still T = 908.60 mK
MC T = 50.21 mK
Still T = 921.20 mK
MC T = 50.20 mK
Still T = 929.00 mK
MC T = 50.24 mK
Still T = 935.70 mK
MC T = 50.20 mK
Still T = 943.70 mK
MC T = 50.24 mK
Still T = 950.00 mK
MC T = 50.17 mK
Still T = 954.80 mK
MC T = 50.21 mK
Still T = 954.50 mK
MC T = 49.97 mK
Still T = 949.10 mK
MC T = 49.93 mK
Still T = 948.10 mK
MC T = 49.75 mK
Still T = 944.50 mK
MC T = 49.84 mK
Still T = 944.50 mK
MC T = 49.90 mK
Still T = 944.90 mK
MC T = 50.01 mK
Still T = 946.50 mK
MC T = 50.12 mK
Still T = 94

In [4]:
# change MC Temperature setpoint
# change still heater setpoint
try:
    ph.set_mc_temperature(decs_visa, 50e-3)  # example: 100 mK
    # ph.set_still_heater_power(decs_visa, 10e-3)  # example: 100 uW

    for i in range(10):
        T_mK = float(decs_visa.query("get_MC_T")) * 1000
        T_still_mK = float(decs_visa.query("get_STILL_T")) * 1000
        # flow = float(decs_visa.query("get_3He_F"))
        print(f"MC T = {T_mK:.2f} mK")
        print(f"Still T = {T_still_mK:.2f} mK")
        # print(f"He3 Flow = {flow:.2f}")
        time.sleep(30)

finally:
    ph.all_heaters_off(decs_visa)

MC T = 11.04 mK
Still T = 690.50 mK
MC T = 11.70 mK
Still T = 689.30 mK
MC T = 23.24 mK
Still T = 687.50 mK
MC T = 33.56 mK
Still T = 686.00 mK
MC T = 37.69 mK
Still T = 684.30 mK
MC T = 40.65 mK
Still T = 682.90 mK
MC T = 43.55 mK
Still T = 681.70 mK
MC T = 45.85 mK
Still T = 680.30 mK
MC T = 47.80 mK
Still T = 679.10 mK
MC T = 49.86 mK
Still T = 678.00 mK


In [6]:
# set a range of MC Temperature setpoint
# change still heater setpoint
repeat_n = 10
mc_setpoint = 5E-3
try:
    ph.set_mc_temperature(decs_visa, mc_setpoint)  # example: 100 mK
    # ph.set_still_heater_power(decs_visa, 10e-3)  # example: 100 uW
    
    for i in range(repeat_n):
        T_mK = float(decs_visa.query("get_MC_T")) * 1000
        T_still_mK = float(decs_visa.query("get_STILL_T")) * 1000
        T_sp_mK = mc_setpoint * 1000
        mc_setpoint = mc_setpoint + (i*1E-3)
        ph.set_mc_temperature(decs_visa, mc_setpoint)
        # flow = float(decs_visa.query("get_3He_F"))
        print(f"Change MC temperature to: {T_sp_mK} mK")
        print(f"MC T = {T_mK:.2f} mK")
        print(f"Still T = {T_still_mK:.2f} mK")
        # print(f"He3 Flow = {flow:.2f}")
        time.sleep(120)

finally:
    ph.all_heaters_off(decs_visa)

Change MC temperature to: 0.005
MC T = 8.53 mK
Still T = 639.80 mK
Change MC temperature to: 0.005
MC T = 8.40 mK
Still T = 639.40 mK
Change MC temperature to: 0.006
MC T = 8.27 mK
Still T = 638.90 mK
Change MC temperature to: 0.008
MC T = 8.13 mK
Still T = 638.50 mK
Change MC temperature to: 0.011
MC T = 8.02 mK
Still T = 638.20 mK
Change MC temperature to: 0.015
MC T = 8.10 mK
Still T = 637.90 mK
Change MC temperature to: 0.02
MC T = 19.81 mK
Still T = 636.20 mK
Change MC temperature to: 0.026000000000000002
MC T = 29.91 mK
Still T = 635.10 mK
Change MC temperature to: 0.033
MC T = 33.78 mK
Still T = 634.40 mK
Change MC temperature to: 0.041
MC T = 41.12 mK
Still T = 632.60 mK


In [4]:
# set temperature
decs_visa.query("set_MC_T:0.050")

'0.05'

In [ ]:
# get still temperature
decs_visa.query("get_STILL_T")

In [ ]:
# set still heater
decs_visa.query("set_STILL_H:0.002")

In [ ]:
# set MC heater
decs_visa.query("set_MC_H:0.000012")

In [ ]:
# set heaters off
decs_visa.query("set_MC_H_OFF:0")
decs_visa.query("set_STILL_H_OFF:0")

**NB** If the query above threw and exception you'll need to check the server settings, and the DECS<->VISA logs to determine why...

Assuming the above worked correctly; run a simple loop to pull back some data from the system:

In [ ]:
import matplotlib.pyplot as plt

pt2_temps = []
time_stamps = []
for i in range(20):
    time_stamps.append(time.time())
    pt2_temps.append(float(decs_visa.query("get_PT2_T1").strip()))
    time.sleep(1)

fig, ax = plt.subplots()
fig.set_facecolor("darkgrey")
ax.set_facecolor('black')
plt.xlabel(r'Elapsed time [s]')
plt.ylabel(r'PT2 temperature [K]')
plt.grid(visible=True, which = 'major', color = 'grey', linestyle = '--')
plt.scatter(time_stamps, pt2_temps)
plt.show()

We'll want to stop the DECS<->VISA process once we're done, so

In [7]:
decs_visa.write("SHUTDOWN")

9

In [ ]:
# set PRXS to Bypass off circulation
# TP speed will start to slow down when flow is high where still pressure rises above 2
# Check status of CP and bypass valve
